## E-commerce & Retail Analytics

In [1]:
# Load the CSV files using Pandas

import pandas as pd

orders = pd.read_csv("data/olist_orders_dataset.csv")
items = pd.read_csv("data/olist_order_items_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
customers = pd.read_csv("data/olist_customers_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
translation = pd.read_csv("data/product_category_name_translation.csv")

#### Check the structure and size of the data

In [2]:
orders.shape

(99441, 8)

In [3]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [4]:
orders.isnull().sum().sort_values(ascending=False).head(10)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

In [5]:
# Provides statistical summaries of numerical columns

orders.describe(include='all')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [6]:
# Check the duplicate value

orders.duplicated().sum()

np.int64(0)

#### Clean the dates and prepare the data

In [7]:
# Convert them to proper datetime values

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"],
    errors="coerce"
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"],
    errors="coerce"
)

orders["order_purchase_timestamp"], orders["order_delivered_customer_date"], orders["order_estimated_delivery_date"]

(0       2017-10-02 10:56:33
 1       2018-07-24 20:41:37
 2       2018-08-08 08:38:49
 3       2017-11-18 19:28:06
 4       2018-02-13 21:18:39
                 ...        
 99436   2017-03-09 09:54:05
 99437   2018-02-06 12:58:58
 99438   2017-08-27 14:46:43
 99439   2018-01-08 21:28:27
 99440   2018-03-08 20:57:30
 Name: order_purchase_timestamp, Length: 99441, dtype: datetime64[us],
 0       2017-10-10 21:25:13
 1       2018-08-07 15:27:45
 2       2018-08-17 18:06:29
 3       2017-12-02 00:28:42
 4       2018-02-16 18:17:02
                 ...        
 99436   2017-03-17 15:08:01
 99437   2018-02-28 17:37:56
 99438   2017-09-21 11:24:17
 99439   2018-01-25 23:32:54
 99440   2018-03-16 13:08:30
 Name: order_delivered_customer_date, Length: 99441, dtype: datetime64[us],
 0       2017-10-18
 1       2018-08-13
 2       2018-09-04
 3       2017-12-15
 4       2018-02-26
            ...    
 99436   2017-03-28
 99437   2018-03-02
 99438   2017-09-27
 99439   2018-02-15
 99440   2018-0

In [8]:
# create the late-delivery indicator

orders["late_delivery"] = (
    orders["order_status"].eq("delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
    & (
        orders["order_delivered_customer_date"]
        > orders["order_estimated_delivery_date"]
    )
).astype(int)

orders["late_delivery"]

0        0
1        0
2        0
3        0
4        0
        ..
99436    0
99437    0
99438    0
99439    0
99440    0
Name: late_delivery, Length: 99441, dtype: int64

#### Join the tables

In [9]:
# prepare review scores at the order level

order_reviews = (
    reviews.groupby("order_id")
    .agg(review_score=("review_score", "mean"))
    .reset_index()
)

order_reviews

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0
98669,fffcd46ef2263f404302a634eb57f7eb,5.0
98670,fffce4705a9662cd70adb13d4a31832d,5.0
98671,fffe18544ffabc95dfada21779c9644f,5.0


In [10]:
# Combine the orders, reviews, and customer information into an order-level table

orders_analysis = (
    orders
    .merge(order_reviews, on="order_id", how="left")
    .merge(
        customers[["customer_id", "customer_state"]],
        on="customer_id",
        how="left"
    )
)

orders_analysis.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,late_delivery,review_score,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,4.0,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,4.0,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,5.0,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,5.0,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,5.0,SP


In [11]:
# Combine the item, product, seller and order information

sales_analysis = (
    items
    .merge(
        orders_analysis,
        on="order_id",
        how="left"
    )
    .merge(products, on="product_id", how="left")
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id",
        how="left"
    )
    .merge(
        translation,
        on="product_category_name",
        how="left"
    )
)

sales_analysis[
    [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "customer_state",
        "product_category_name_english",
        "review_score"
    ]
].head()

,order_id,product_id,seller_id,price,freight_value,customer_state,product_category_name_english,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29,RJ,cool_stuff,5.0
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93,SP,pet_shop,4.0
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87,MG,furniture_decor,5.0
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,SP,perfumery,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14,SP,garden_tools,5.0


#### Business Question 1: Which product categories and regions generate the most revenue?
##### Business Problem
##### Management wants to know which product categories and customer regions contribute the most sales revenue.

In [13]:
# Revenue by product category

category_revenue = (
    sales_analysis
    .groupby("product_category_name_english", dropna=False)
    .agg(
        total_revenue=("price", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

category_revenue.head(10)

,product_category_name_english,total_revenue,total_items
43,health_beauty,1258681.34,9670
70,watches_gifts,1205005.68,5991
7,bed_bath_table,1036988.68,11115
65,sports_leisure,988048.97,8641
15,computers_accessories,911954.32,7827
39,furniture_decor,729762.49,8334
20,cool_stuff,635290.85,3796
49,housewares,632248.66,6964
5,auto,592720.11,4235
42,garden_tools,485256.46,4347


In [12]:
# Revenue by customer region

regional_revenue = (
    sales_analysis
    .groupby("customer_state")
    .agg(
        total_revenue=("price", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

regional_revenue

,customer_state,total_revenue,total_items
25,SP,5202955.05,47449
18,RJ,1824092.67,14579
10,MG,1585308.03,13129
22,RS,750304.02,6235
17,PR,683083.76,5740
23,SC,520553.34,4176
4,BA,511349.99,3799
6,DF,302603.94,2406
8,GO,294591.95,2333
7,ES,275037.31,2256


#### Business Question 2: How much revenue is generated by each seller and product category?
##### Business Problem
##### Management wants to identify the sellers and product categories contributing the most revenue.

In [14]:
# creates a table showing revenue for every seller-category combination

seller_category_revenue = (
    sales_analysis
    .groupby(
        ["seller_id", "product_category_name_english"],
        dropna=False
    )
    .agg(
        total_revenue=("price", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

seller_category_revenue.head(20)

,seller_id,product_category_name_english,total_revenue,total_items
1849,4869f7a5dfa277a7dca6462dcf3b52b2,watches_gifts,201071.73,1002
6454,fa1c13f2614d7b5c4749cbc52fecda94,watches_gifts,192092.74,579
3287,7c67e1448b00f6e969d365cea6b010ab,office_furniture,172959.66,1233
3357,7e93a43ef30c4f03f38b393420bc753a,watches_gifts,169768.06,314
2173,53243585a1d6dc2643021fd1853d8905,computers,166871.75,153
1892,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,165219.25,1572
5592,da8622b14eb17ae2831f4ac5b9dab84a,bed_bath_table,152307.80,1277
3228,7a67c85e85bb2ce8582c35f2203ad736,cool_stuff,135413.55,1069
405,1025f0e2d44d7041d6cf58b6550e0bfa,furniture_decor,111530.45,1292
812,1f50f920176fa81dab994f9023523100,garden_tools,103133.11,1882


In [15]:
# Find the overall top sellers

seller_revenue = (
    sales_analysis
    .groupby("seller_id")
    .agg(
        total_revenue=("price", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

seller_revenue.head(10)

,seller_id,total_revenue,total_items
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1156
1013,53243585a1d6dc2643021fd1853d8905,222776.05,410
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1987
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,586
1535,7c67e1448b00f6e969d365cea6b010ab,187923.89,1364
1560,7e93a43ef30c4f03f38b393420bc753a,176431.87,340
2643,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1551
1505,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1171
192,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,1428
1824,955fee9216a65b617aa5c0531780ce60,135171.70,1499


#### Business Question 3: What percentage of orders arrive later than the estimated delivery date?
##### Business Problem
##### Late deliveries can indicate fulfilment or logistics issues and may affect customer satisfaction.

In [16]:
# Use the order-level table so each order is counted once, even if it contains multiple items

eligible_deliveries = orders_analysis.loc[
    (orders_analysis["order_status"] == "delivered")
    & orders_analysis["order_delivered_customer_date"].notna()
    & orders_analysis["order_estimated_delivery_date"].notna()
]

late_delivery_rate = (
    eligible_deliveries["late_delivery"].mean() * 100
)

late_delivery_rate

np.float64(8.112366538820359)

In [17]:
# Calculate the number of late and on-time orders

delivery_summary = (
    eligible_deliveries
    .groupby("late_delivery")
    .agg(total_orders=("order_id", "nunique"))
    .reset_index()
)

delivery_summary

,late_delivery,total_orders
0,0,88644
1,1,7826


In [18]:
# Calculate the late-delivery rate by customer state

regional_delivery = (
    eligible_deliveries
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "nunique"),
        late_delivery_rate=("late_delivery", "mean")
    )
    .reset_index()
)

regional_delivery["late_delivery_rate_pct"] = (
    regional_delivery["late_delivery_rate"] * 100
)

regional_delivery.sort_values(
    "late_delivery_rate_pct",
    ascending=False
)

,customer_state,total_orders,late_delivery_rate,late_delivery_rate_pct
1,AL,397,0.239295,23.929471
9,MA,717,0.196653,19.665272
16,PI,476,0.159664,15.966387
5,CE,1279,0.153245,15.324472
24,SE,335,0.152239,15.223881
4,BA,3256,0.140356,14.035627
18,RJ,12350,0.134737,13.473684
26,TO,274,0.127737,12.773723
13,PA,946,0.123679,12.367865
7,ES,1995,0.122306,12.230576


#### Business Question 4: Is late delivery associated with lower customer review scores?
##### Business Problem
##### Management wants to understand whether customers give lower ratings to orders that arrive later than estimated.

In [19]:
# Compares the average review score of late orders with the average score of on-time orders

delivery_review_analysis = (
    eligible_deliveries
    .dropna(subset=["review_score"])
    .groupby("late_delivery")
    .agg(
        total_orders=("order_id", "nunique"),
        average_review_score=("review_score", "mean")
    )
    .reset_index()
)

delivery_review_analysis

,late_delivery,total_orders,average_review_score
0,0,88163,4.294292
1,1,7661,2.566506


In [20]:
# Calculate the difference in review scores

review_comparison = (
    delivery_review_analysis
    .set_index("late_delivery")["average_review_score"]
)

review_score_difference = (
    review_comparison.loc[1] - review_comparison.loc[0]
)

review_score_difference

np.float64(-1.7277863340342652)

#### Business Question 5: Which product categories have high freight costs or poor customer satisfaction?
##### Business Problem
##### A category may generate strong revenue but still have high delivery costs or poor customer reviews.

In [21]:
# Freight costs by category

category_freight = (
    sales_analysis
    .groupby("product_category_name_english", dropna=False)
    .agg(
        total_freight=("freight_value", "sum"),
        average_freight=("freight_value", "mean"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
    .sort_values("average_freight", ascending=False)
)

category_freight.head(10)

,product_category_name_english,total_freight,average_freight,total_items
14,computers,9836.30,48.454680,203
45,home_appliances_2,10600.18,44.538571,238
41,furniture_mattress_and_upholstery,1630.46,42.906842,38
51,kitchen_dining_laundry_garden_furniture,11999.43,42.702598,281
38,furniture_bedroom,4632.23,42.497523,109
57,office_furniture,68571.95,40.551124,1691
64,small_appliances_home_oven_and_coffee,2747.86,36.156053,76
40,furniture_living_room,17968.17,35.722008,503
62,signaling_and_security,6507.82,32.702613,199
50,industry_commerce_and_business,7884.68,29.420448,268


In [22]:
# Customer satisfaction by category

category_reviews = (
    sales_analysis[
        [
            "order_id",
            "product_category_name_english",
            "review_score"
        ]
    ]
    .drop_duplicates(
        ["order_id", "product_category_name_english"]
    )
    .groupby("product_category_name_english", dropna=False)
    .agg(
        average_review_score=("review_score", "mean"),
        reviewed_orders=("order_id", "nunique")
    )
    .reset_index()
    .sort_values("average_review_score")
)

category_reviews.head(10)

,product_category_name_english,average_review_score,reviewed_orders
61,security_and_services,2.500000,2
57,office_furniture,3.616390,1273
30,fashion_male_clothing,3.702703,112
27,fashio_female_clothing,3.730769,39
23,diapers_and_hygiene,3.740741,27
41,furniture_mattress_and_upholstery,3.815789,38
46,home_comfort_2,3.826087,24
4,audio,3.834294,350
19,construction_tools_safety,3.849398,167
47,home_confort,3.863291,397


In [23]:
# Combine the findings

category_performance = (
    category_revenue
    .merge(
        category_freight,
        on="product_category_name_english",
        how="left"
    )
    .merge(
        category_reviews,
        on="product_category_name_english",
        how="left"
    )
)

category_performance.sort_values(
    ["average_freight", "average_review_score"],
    ascending=[False, True]
)

,product_category_name_english,total_revenue,total_items_x,total_freight,average_freight,total_items_y,average_review_score,reviewed_orders
16,computers,222963.13,203,9836.30,48.454680,203,4.174157,181
26,home_appliances_2,113317.74,238,10600.18,44.538571,238,4.133621,234
61,furniture_mattress_and_upholstery,4368.08,38,1630.46,42.906842,38,3.815789,38
37,kitchen_dining_laundry_garden_furniture,46328.37,281,11999.43,42.702598,281,4.048780,248
48,furniture_bedroom,20028.78,109,4632.23,42.497523,109,4.131579,95
...,...,...,...,...,...,...,...,...
41,food,29393.41,510,7271.03,14.256922,510,4.276404,450
68,home_comfort_2,760.27,30,410.31,13.677000,30,3.826087,24
62,fashio_female_clothing,2803.64,48,621.75,12.953125,48,3.730769,39
59,books_imported,4639.85,60,769.85,12.830833,60,4.320755,53


In [24]:
# Exporting from Pandas

sales_analysis.to_csv(
    "data/sales_analysis.csv",
    index=False
)

In [25]:
orders_analysis.to_csv(
    "data/orders_analysis.csv",
    index=False
)